# Spring 2025-2026 CS 412 Recitation 1: Prototypes and k-NN Using the Iris Dataset

## Google Colab

Google Colab is a **cloud-based Jupyter notebook** environment that lets you write and run Python code directly in your browser.

- **Cells**: notebooks consist of code cells and text cells. You can run a code cell by either pressing the **▶ button** or with **Shift+Enter**.
- **Runtime**: you're running on Google's servers. If you're idle for too long, the session may disconnect.
- **Temporary storage**: files you create during a session are lost when it ends. Save your notebook to Google Drive to keep your work.
- You can use a free GPU for a limited amount of time. You can use this GPU by following **Edit --> Notebook Settings --> Select Hardware accelerator as 'T4 GPU'**. (All though you would not need to set this for this tutorial as the dataset we will be using is very small)

# 🌸 The Iris Dataset

The Iris dataset is one of the most classic datasets in machine learning. It contains 150 samples of iris flowers, each belonging to one of 3 species:

- Iris Setosa
- Iris Versicolor
- Iris Virginica

Each sample has 4 features:
- Sepal length
- Sepal width
- Petal length
- Petal width

The goal is to **classify** a flower into one of the 3 species based on these measurements.

In [ ]:
import numpy as np                                    # numerical computing, arrays and math operations
import pandas as pd                                   # data manipulation and analysis, works with tables (DataFrames)
import matplotlib.pyplot as plt                       # basic plotting and visualization
import seaborn as sns                                 # statistical data visualization, built on top of matplotlib
from sklearn.datasets import load_iris                # loads the built-in Iris dataset from scikit-learn

In [ ]:
# Load dataset
iris = load_iris()

In [ ]:
type(iris)

In [ ]:
iris

In [ ]:
iris.target

In [ ]:
iris.target_names

In [ ]:
df = pd.DataFrame(iris.data, columns=iris.feature_names)                    # Features
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)   # Class labels

In [ ]:
# Preview first few rows
print("First 5 rows:")
display(df.head())

In [ ]:
# Class distribution
print("\nClass Distribution:")
print(df['species'].value_counts())

# Visualize class distribution
plt.figure(figsize=(6,4))
sns.countplot(x='species', data=df, palette='Set2')
plt.title('Class Distribution')
plt.xlabel('Species')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feature in enumerate(iris.feature_names):
    sns.violinplot(x='species', y=feature, data=df, palette='Set2', ax=axes[i]) # y=feature tells seaborn to look up that column in df and use its numerical values as the y-axis of the violin plot.
    axes[i].set_title(f'Distribution of {feature}')
    axes[i].set_xlabel('Species')
    axes[i].set_ylabel(feature)

plt.suptitle('Feature Distributions by Species', fontsize=14)
plt.tight_layout()
plt.show()

# Classification Methods

1. Prototype Classifier
2. k-NN (k Nearest Neighbor)

In [ ]:
from sklearn.model_selection import train_test_split  # will be used to split our dataset

X = df[iris.feature_names].values
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # stratify --> If the original dataset is 33% setosa, 33% versicolor, 33% virginica,
                                                                                                      #               both the train and test sets will also be roughly 33/33/33

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

## 1. Protoype Classifier

In [ ]:
# Prototype classifier: compute the centroid (mean) of each class
# and set the prediction as the closest centroid's class

def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def compute_prototypes(X_train, y_train):
    classes = np.unique(y_train)
    prototypes = {}
    for c in classes:
        prototypes[c] = X_train[y_train == c].mean(axis=0) # axis=0 means "take the mean along the rows". so it collapses all the rows down into a single row by averaging each column independently.
    return prototypes

def prototype_predict(prototypes, X_test):
    predictions = []
    for test_sample in X_test:
        distances = {c: euclidean_distance(test_sample, proto) for c, proto in prototypes.items()}
        predictions.append(min(distances, key=distances.get)) # finds the key (class number) whose value (distance) is smallest
    return np.array(predictions)

prototypes = compute_prototypes(X_train, y_train)

# Print prototypes
for c, proto in prototypes.items():
    print(f"Prototype for class '{iris.target_names[c]}': {proto.round(2)}")

proto_predictions = prototype_predict(prototypes, X_test)

# Accuracy
proto_accuracy = np.mean(proto_predictions == y_test) # proto_predictions                     = [0, 1, 2, 2, 0]
                                                      # y_test                                = [0, 1, 2, 2, 1]
                                                      # proto_predictions == y_test           = [True, True, True, True, False]
                                                      # np.mean(proto_predictions == y_test)  = 4/5 = 80%
print(proto_predictions)
print(y_test)
print(proto_predictions == y_test)
print(f"\nPrototype Classifier Accuracy: {proto_accuracy:.2f}")

In [ ]:
# Visualize prototypes vs training samples using 2 features (petal length vs petal width)
# for easy 2D visualization.
# Note: In the models we are using all 4 features. We use 2 features just for ease of visulization

feature_x, feature_y = 2, 3  # petal length, petal width
colors = ['#66c2a5', '#fc8d62', '#8da0cb']

plt.figure(figsize=(8, 6))

for i, class_name in enumerate(iris.target_names):
    # plot training samples
    plt.scatter(X_train[y_train == i, feature_x], X_train[y_train == i, feature_y],
                color=colors[i], alpha=0.4, label=f'{class_name} (samples)')
    # plot prototype
    plt.scatter(prototypes[i][feature_x], prototypes[i][feature_y],
                color=colors[i], edgecolors='black', s=200, marker='*',
                label=f'{class_name} (prototype)')
    plt.scatter(X_test[y_test == i, feature_x], X_test[y_test == i, feature_y],
                color=colors[i], alpha=0.6, label=f'{class_name} (test samples)', edgecolors='red')

plt.xlabel(iris.feature_names[feature_x])
plt.ylabel(iris.feature_names[feature_y])
plt.title('Training Samples and Prototypes (Petal Length vs Petal Width)')
plt.legend()
plt.tight_layout()
plt.show()

## 2. k-NN (k Nearest Neighbor)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Now we generalize to k neighbors

k = 5
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train, y_train)
knn_predictions = knn.predict(X_test)

knn_accuracy = np.mean(knn_predictions == y_test)
print(f"kNN (k={k}) Accuracy: {knn_accuracy:.2f}")

### Cross Validation

In [ ]:
from sklearn.model_selection import cross_val_score

# Since our dataset is small, cross-validation gives a more reliable estimate
# of model performance than a single train/test split

# 5-Fold Cross Validation works as follows:
# Split the dataset into 5 equal portions:
# [fold1 | fold2 | fold3 | fold4 | fold5]
#
# Round 1: [fold2 | fold3 | fold4 | fold5] --> Train  |  [fold1] --> Test
# Round 2: [fold1 | fold3 | fold4 | fold5] --> Train  |  [fold2] --> Test
# Round 3: [fold1 | fold2 | fold4 | fold5] --> Train  |  [fold3] --> Test
# Round 4: [fold1 | fold2 | fold3 | fold5] --> Train  |  [fold4] --> Test
# Round 5: [fold1 | fold2 | fold3 | fold4] --> Train  |  [fold5] --> Test
#
# Final score = mean of all 5 test scores

knn_cv = KNeighborsClassifier(n_neighbors=5)
cv_scores = cross_val_score(knn_cv, X_train, y_train, cv=5)

print(f"Cross-validation scores (5-fold): {cv_scores.round(2)}")
print(f"Mean accuracy: {cv_scores.mean():.2f} ± {cv_scores.std():.2f}") # Its good practive to show std_dev, this show how stable results you get accross different splits

In [ ]:
# We can also search for the best k value using cross-validation
# Instead of fixing k=5, let's try a range of k values and pick the best one

k_values = range(1, 21)  # Try k from 1 to 20
cv_mean_scores = []

for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn_k, X_train, y_train, cv=5)
    cv_mean_scores.append(scores.mean())

best_k = k_values[np.argmax(cv_mean_scores)]
print(f"Best k: {best_k} with mean CV accuracy: {max(cv_mean_scores):.2f}")

# Visualize how accuracy changes with different k values
plt.figure(figsize=(8, 4))
plt.plot(k_values, cv_mean_scores, marker='o')
plt.axvline(x=best_k, color='r', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k (Number of Neighbors)')
plt.ylabel('Mean CV Accuracy')
plt.title('KNN Accuracy vs. k Value (5-Fold CV)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Now let's train a final kNN model using the best k we found
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
knn_predictions = knn_best.predict(X_test)

print(f"Final model accuracy with best k={best_k}: {np.mean(knn_predictions == y_test):.2f}")

# Measuring Performance

## Conufiosn Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, knn_predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)

disp.plot(cmap='Blues')
plt.title(f'Confusion Matrix — kNN (k={best_k})')
plt.tight_layout()
plt.show()

## Performance Metrics

Once we have our predictions, we need to measure how well our model performed. Simply using accuracy not always enough. We need more detailed metrics.

---

### Precision
Out of all the samples we **predicted** as class X, how many were **actually** class X?

$$Precision = \frac{TP}{TP + FP}$$

A high precision means: when we say it's class X, we're usually right.

---

### Recall
Out of all the samples that **actually belong** to class X, how many did we **correctly predict** as class X?

$$Recall = \frac{TP}{TP + FN}$$

A high recall means: we're good at catching all the actual class X samples.

---

### F1 Score
The **harmonic mean** of precision and recall. Balances both metrics into a single score.

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

Useful when you want a single metric that captures both precision and recall.

---

### Support
The **number of actual samples** in each class in the test set. Not a performance metric — just tells you how many samples each score is based on.

---

### Averages

**Macro average**: computes the metric for each class independently and takes the **unweighted mean**. Treats all classes equally regardless of size.

$$Macro\ Avg = \frac{metric_{class1} + metric_{class2} + metric_{class3}}{3}$$

**Weighted average**: same as macro but weights each class by its **support** (number of samples). Gives more importance to larger classes.

$$Weighted\ Avg = \frac{\sum (metric_{class_i} \times support_{class_i})}{\sum support_{class_i}}$$

---

*Note: TP = True Positive, FP = False Positive, FN = False Negative*

In [ ]:
from sklearn.metrics import classification_report

print(f"Classification Report — kNN (k={best_k}):")
print(classification_report(y_test, knn_predictions, target_names=iris.target_names))